# Financial Analyst

> Notebook generated from [`examples/subgraphs/02_financial_analyst.py`](../../examples/subgraphs/02_financial_analyst.py).

| Item | Value |
|------|-------|
| **Dataset** | Yahoo Finance NVDA, MSFT (embedded snapshots) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** market_data → technical → fundamental → risk → [3 gates] → report. BUY/HOLD/SELL decision + risk score.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Financial Analyst Subgraph — Complete stock market analysis
======================================================================
Subgraph: prismal.agents.subgraphs.financial

Dataset: Yahoo Finance / Alpha Vantage (real market data)
  • Ticker symbols: NVDA, MSFT, AAPL, GOOGL (2023-2024)
  • Reference: yfinance API + free Alpha Vantage
  • Why: The Financial subgraph has 5 specialized agents and 3
    quality gates. Large-cap tech stock data
    is ideal for demonstrating technical, fundamental, and risk analysis.

Financial Analyst subgraph description:
  market_data_collector → technical_analyst → fundamental_analyst
  → risk_sentiment_analyst → [optional HITL gate] → report_generator

  Quality Gates:
  1. Data Gate     : after market_data_collector — skips if confidence < threshold
  2. Technical Gate: after technical_analyst — skips fundamental if too noisy
  3. HITL Gate     : after report_generator — requires human approval (optional)

Nodes:
  1. market_data_collector  — prices, volume, historical data
  2. technical_analyst      — chart patterns, momentum indicators
  3. fundamental_analyst    — P/E, earnings growth, balance sheet
  4. risk_sentiment_analyst — VIX, beta, market sentiment
  5. report_generator       — synthesis with BUY/HOLD/SELL recommendation

Usage:
    uv run python examples/subgraphs/02_financial_analyst.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.messages import HumanMessage

from prismal.agents.state import create_initial_state

# Import with error handling in case the subgraph is not registered
try:
    from prismal.agents.subgraphs.financial.builder import (
        get_compiled_financial_analyst,
        register_financial_analyst,
    )

    FINANCIAL_SUBGRAPH_AVAILABLE = True
except ImportError:
    FINANCIAL_SUBGRAPH_AVAILABLE = False

## Dataset: tickers and metadata

In [ ]:
ANALYSIS_REQUESTS = [
    {
        "ticker": "NVDA",
        "company": "NVIDIA Corporation",
        "sector": "Semiconductors",
        "analysis_date": "2024-01-15",
        "timeframe": "6M",
        "analysis_focus": "AI chip demand and data center growth",
        "context": (
            "NVIDIA has experienced massive growth driven by demand "
            "for GPUs used in AI model training. The stock rose >200% "
            "in 2023. Analyze whether momentum will continue or whether there is correction risk."
        ),
    },
    {
        "ticker": "MSFT",
        "company": "Microsoft Corporation",
        "sector": "Software / Cloud",
        "analysis_date": "2024-01-15",
        "timeframe": "1Y",
        "analysis_focus": "Azure AI integration and Copilot monetization",
        "context": (
            "Microsoft has integrated AI across all its products with GitHub Copilot, "
            "Azure OpenAI Service and Microsoft 365 Copilot. Assess the impact on "
            "valuation and revenue growth prospects."
        ),
    },
]

## Simulated market data (format the agent would use)

In [ ]:
MOCK_MARKET_DATA = {
    "NVDA": {
        "current_price": 495.22,
        "52w_high": 505.48,
        "52w_low": 108.13,
        "market_cap": "1.22T",
        "volume_avg": "48.2M",
        "pe_ratio": 65.3,
        "eps_ttm": 7.58,
        "revenue_growth_yoy": 122.4,
        "gross_margin": 73.8,
        "beta": 1.74,
        "rsi_14": 62.3,
        "macd_signal": "bullish_crossover",
        "support_levels": [450, 420, 390],
        "resistance_levels": [500, 520, 550],
        "analyst_consensus": "BUY",
        "price_targets": {"low": 400, "median": 550, "high": 700},
    },
    "MSFT": {
        "current_price": 374.51,
        "52w_high": 384.30,
        "52w_low": 219.35,
        "market_cap": "2.78T",
        "volume_avg": "22.1M",
        "pe_ratio": 36.2,
        "eps_ttm": 10.34,
        "revenue_growth_yoy": 16.0,
        "gross_margin": 69.8,
        "beta": 0.89,
        "rsi_14": 58.1,
        "macd_signal": "neutral",
        "support_levels": [360, 340, 315],
        "resistance_levels": [380, 395, 420],
        "analyst_consensus": "BUY",
        "price_targets": {"low": 325, "median": 410, "high": 480},
    },
}


def format_analysis_request(req: dict) -> str:
    """Format the analysis request as a message for the agent."""
    data = MOCK_MARKET_DATA.get(req["ticker"], {})

    return (
        f"Analyze the {req['ticker']} ({req['company']}) stock for the date "
        f"{req['analysis_date']}.\n\n"
        f"Sector: {req['sector']}\n"
        f"Analysis period: {req['timeframe']}\n"
        f"Focus: {req['analysis_focus']}\n\n"
        f"Context:\n{req['context']}\n\n"
        f"Available market data:\n"
        f"  Current price: ${data.get('current_price', 'N/A')}\n"
        f"  Market cap: {data.get('market_cap', 'N/A')}\n"
        f"  P/E Ratio: {data.get('pe_ratio', 'N/A')}\n"
        f"  Revenue growth YoY: {data.get('revenue_growth_yoy', 'N/A')}%\n"
        f"  Beta: {data.get('beta', 'N/A')}\n"
        f"  RSI(14): {data.get('rsi_14', 'N/A')}\n"
        f"  MACD: {data.get('macd_signal', 'N/A')}\n"
        f"  Analyst consensus: {data.get('analyst_consensus', 'N/A')}\n\n"
        f"Generate a complete analysis with a BUY/HOLD/SELL recommendation and price target."
    )


async def run_financial_analysis(request: dict) -> None:
    """Run the financial analysis subgraph.

    Args:
        request: Analysis request with ticker and metadata.
    """
    print(f"\n[Analysis: {request['ticker']} — {request['company']}]")
    print(f"  Sector : {request['sector']}")
    print(f"  Date   : {request['analysis_date']}")
    print(f"  Focus  : {request['analysis_focus']}")

    if not FINANCIAL_SUBGRAPH_AVAILABLE:
        # Demo mode: show what the subgraph would do
        print("\n  [Demo mode — simulated subgraph]")
        market_data = MOCK_MARKET_DATA.get(request["ticker"], {})

        print("\n  [Node 1: market_data_collector]")
        print(f"    Price: ${market_data.get('current_price')}")
        print(f"    Cap  : {market_data.get('market_cap')}")
        print("    ✓ Data Gate: high confidence → proceed")

        print("\n  [Node 2: technical_analyst]")
        print(
            f"    RSI(14): {market_data.get('rsi_14')} ({'overbought' if market_data.get('rsi_14', 0) > 70 else 'neutral'})"
        )
        print(f"    MACD: {market_data.get('macd_signal')}")
        sups = market_data.get("support_levels", [])
        resis = market_data.get("resistance_levels", [])
        print(f"    Supports: {sups[:2]}")
        print(f"    Resistances: {resis[:2]}")
        print("    ✓ Technical Gate: clear signals → proceed to fundamental")

        print("\n  [Node 3: fundamental_analyst]")
        print(f"    P/E Ratio: {market_data.get('pe_ratio')} (sector avg: ~30)")
        print(f"    EPS TTM: ${market_data.get('eps_ttm')}")
        print(f"    Revenue growth: {market_data.get('revenue_growth_yoy')}% YoY")
        print(f"    Gross Margin: {market_data.get('gross_margin')}%")

        print("\n  [Node 4: risk_sentiment_analyst]")
        beta = market_data.get("beta", 1.0)
        risk_level = "HIGH" if beta > 1.5 else "MEDIUM" if beta > 1.0 else "LOW"
        print(f"    Beta: {beta} → Risk relative to market: {risk_level}")
        print(
            f"    Price/52w High: {(market_data.get('current_price', 0) / market_data.get('52w_high', 1) * 100):.1f}%"
        )

        print("\n  [Node 5: report_generator]")
        consensus = market_data.get("analyst_consensus", "HOLD")
        target_med = market_data.get("price_targets", {}).get("median", 0)
        current = market_data.get("current_price", 0)
        upside = ((target_med - current) / current * 100) if current > 0 else 0
        print(f"    Recommendation: {consensus}")
        print(f"    Price target (median): ${target_med}")
        print(f"    Upside potential: {upside:+.1f}%")
        print(f"    Analyst targets: {market_data.get('price_targets')}")
        return

    # Real mode with subgraph
    await register_financial_analyst()
    subgraph = await get_compiled_financial_analyst()

    state = create_initial_state(session_id="nb-financial-analyst")
    state["messages"] = [HumanMessage(content=format_analysis_request(request))]
    state["metadata"] = {
        "ticker": request["ticker"],
        "market_data": MOCK_MARKET_DATA.get(request["ticker"], {}),
        "hitl_enabled": False,  # disable human approval for the example
    }

    config = {"configurable": {"thread_id": f"fin_{request['ticker']}_001"}}
    final_state = await subgraph.ainvoke(state, config=config)

    messages = final_state.get("messages", [])
    if messages:
        print("\n  Generated report:")
        print(f"  {str(messages[-1].content)[:500]}")


async def main() -> None:
    print("=" * 70)
    print("  Financial Analyst Subgraph — Dataset: Yahoo Finance (NVDA, MSFT)")
    print("=" * 70)

    # Subgraph architecture
    print("\n[Financial subgraph architecture]")
    nodes_and_gates = [
        ("market_data_collector", "Collects prices, volume, history"),
        ("[Data Gate]           ", "confidence >= threshold?"),
        ("technical_analyst     ", "RSI, MACD, Bollinger, patterns"),
        ("[Technical Gate]      ", "technical signal clear enough?"),
        ("fundamental_analyst   ", "P/E, EPS, revenue growth, balance sheets"),
        ("risk_sentiment_analyst", "beta, VIX, sentiment"),
        ("[Optional HITL Gate]  ", "requires human approval?"),
        ("report_generator      ", "synthesis + BUY/HOLD/SELL recommendation"),
    ]
    for node, desc in nodes_and_gates:
        print(f"  {node}: {desc}")

    # Run analysis for each ticker
    for request in ANALYSIS_REQUESTS:
        await run_financial_analysis(request)
        print("─" * 70)

    # Additional information about the gates
    print("\n[Subgraph quality gates]")
    print("  Gate 1 (Data Availability):")
    print("    If missing_fields > 0 AND confidence < min_threshold")
    print("    → Skip directly to report (insufficient data)")
    print()
    print("  Gate 2 (Technical Confidence):")
    print("    If technical signal too noisy")
    print("    → Skip fundamental analysis (avoid unfounded analysis)")
    print()
    print("  Gate 3 (HITL):")
    print("    If hitl_enabled=True → pause and wait for human approval")
    print("    Configurable: settings.hitl_enabled = False to skip")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()